In [ ]:
import sqlite3
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import roc_auc_score, confusion_matrix, roc_curve
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
%cd /Volumes/500Gb
%pwd
%ls

In [ ]:
#exploring MAXAGE_VISIBLE_SEX2023
max_con = sqlite3.connect('MAXAGE_VISIBLE_SEX2023.db') #database 
c_m = max_con.cursor()

# Get all table names
c_m.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = c_m.fetchall()

# Print table names
print("Tables in database:")
for table in tables:
    print(f"- {table[0]}")
    print(f"Columns in the table {table[0]}")
    c_m.execute(f"PRAGMA table_info({table[0]});")
    columns = c_m.fetchall()
    print(columns)




In [ ]:
#exploring MAXAGE_VISIBLE_SEX2023
max_con = sqlite3.connect('MAXAGE_VISIBLE_SEX2023.db')
c_m = max_con.cursor()
c_m.execute("""SELECT * FROM AGE_VISIBLE LIMIT 10;""")
results = c_m.fetchall()
print(results)

In [ ]:
#explore total number of patients
c_m.execute("SELECT COUNT(enrolid) FROM AGE_VISIBLE")
all_patients = c_m.fetchone()
print(all_patients)

In [ ]:
print(f"The total number of patients found in MAXAGE_VISIBLE_SEX2023 database is {all_patients[0]}")

In [ ]:
c_m.execute("""SELECT * FROM AGE_VISIBLE" 
"WHERE years_visible >= 1""")

more_than_a_year = c_m.fetchall()
ids = [el_of_tuple[0] for el_of_tuple in more_than_a_year] #collect all the patients ids that fit the criteria of more than 1 year mark
print(len(ids))

In [ ]:
#this is how the data looks like after filtering
for record in more_than_a_year[0:10]:
    print(record)

## Some comments about the data
enrolid - unique patient identifier, it is assigned once and stays the same for an individual patient <br>
maxage - the oldest age the person was observed in data or the age *at the end of the latest enrollments*


In [ ]:
c_m.execute("""SELECT * FROM AGE_VISIBLE
            WHERE years_visible >= 1 AND maxage >= 20""")
year_and_older_20 = c_m.fetchall()


In [ ]:
ids2 = [el_of_tuple[0] for el_of_tuple in year_and_older_20]
print(f"The number of patients who have been enrolled for at least one year and are 20 years old and older are {len(ids2)}")

In [ ]:
%cd /Volumes/FastSSD_2
%pwd
%ls *.db

In [ ]:
#exploring DX_2023,db
dx_con = sqlite3.connect('DX_2023.db') #database 
c_dx = dx_con.cursor()

# Get all table names
c_dx.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables_dx = c_dx.fetchall()

# Print table names
print("Tables in database:")
for table in tables_dx:
    print(f"- {table[0]}")
    print(f"Columns in the table {table[0]}")
    c_dx.execute(f"PRAGMA table_info({table[0]});")
    columns = c_dx.fetchall()
    print(columns)

In [ ]:
c_dx.execute("""SELECT * FROM DX LIMIT 10;""")
results_dx = c_dx.fetchall()
print(results_dx)

## DO NOT LAUNCH UNLESS TRULLY NECESSARY

In [ ]:
c_dx.execute("""SELECT COUNT(enrolid) from DX""")
all_pat_2nd_db = c_dx.fetchone()
print(f"The number of patients in the DX_2023 database {all_pat_2nd_db}")

It took 76 minutes to process this coammnd above, so the total number of patients in the DX_2023.db database is 12,013,523,777

In [ ]:
#c_dx.execute("SELECT COUNT(*) FROM DX")
#perfromed this command in the terminal and it only took about 4 minutes
#the result is 12,013,523,777

# Comments about data
The DX_2023.db contains total of 12 billion of records (12,013,523,777), no enrolid is NAN. <br> 

The total number of patients found in MAXAGE_VISIBLE_SEX2023 database is 16,7301,404 <br>

# Important note
DX_2023.db database might contain numerous duplicate records for enrolid because each disease code **is its own record**. <br>

In [ ]:
#do not run, takes 95.7 min
diabetes_codes = ['249','250','E08','E09','E10','E11','E12','E13']

conditions = ' OR '.join([f"dx LIKE '{code}%'" for code in diabetes_codes]) #try conditions = ' OR '.join([f"SUBSTR(dx, 1, 3) = '{code}'" for code in diabetes_codes])
query = f"SELECT COUNT(enrolid) FROM DX WHERE {conditions}"
c_dx.execute(query)
patients_with_diab = c_dx.fetchone()
print(f"Records from DX_2023.db with diabetes {patients_with_diab[0]}")

The number of records that match diabetic codes is 261,790,645

In [ ]:
pip install pickleshare

In [ ]:
#Lets store all the variables
#Using python in-built method
%store all_patients 
%store more_than_a_year 
%store ids 
%store year_and_older_20 
%store ids2 
%store all_pat_2nd_db 
%store patients_with_diab

In [ ]:
#Lets store all the variables
#Using python in-built method
#all_patients #number of patients in MAXAGE_VISIBLE_SEX2023
#more_than_a_year #number of patients from MAXAGE_VISIBLE_SEX2023 that were enrolled for more than a year
#ids #ids of people that are enrolled for more than a year - list
#year_and_older_20 #number of patients from MAXAGE_VISIBLE_SEX2023 that were enrolled for more than a year and are older than 20 yeras old
#ids2 #ids of people that are enrolled for more than a year and 20 years old and older - list
#all_pat_2nd_db #number of records in DX_2023.db (not patients - patients duplicate)
#patients_with_diab #number of records in DX_2023.db that match the criteria of having diabetes codes - diabetes codes were retrieved using R code from the original author

## DO NOT RUN UNLESS FOR REPRODUCIBILITY - took 146 min

In [ ]:
#now lets see how many patients that have enrol ids from year_and_older20 also have diabetes ICD codes associated with them
conditions2 = ' OR '.join([f"SUBSTR(dx, 1, 3) = '{code}'" for code in diabetes_codes])
query2 = f"SELECT DISTINCT enrolid FROM DX WHERE {conditions2}"

print("Fetching diabetes patient IDs...")
c_dx.execute(query2)
diabetes_patient_ids = [row[0] for row in c_dx.fetchall()]
# Store for later use
%store diabetes_patient_ids


In [ ]:
print(f"Found {len(diabetes_patient_ids)} patients with diabetes")

In [ ]:
diabetes_ids_set = set(diabetes_patient_ids)
one_year_and_older_than20 = set(ids2)

matching_both = diabetes_ids_set & one_year_and_older_than20

print(f"Patients matching both criteria: {len(matching_both)}")
%store matching_both

In [ ]:
import pickle

In [ ]:
variables_to_save = {
    "ids_diabetes_DX2023": diabetes_patient_ids,
    "ids_1year_20_Max": ids2,
    "matching_ids": matching_both,
    "max_records_count":all_patients, 
    "max_1year": more_than_a_year,
    "max_ids_1year":ids,
    "max_records_count_1year_20":year_and_older_20 ,
    "dx2023_records_count":all_pat_2nd_db,
    "dx2023_diabetes_rec_count": patients_with_diab   

}
with open('September24.pkl', 'wb') as f:
    pickle.dump(variables_to_save, f)

In [ ]:
with open('September24.pkl', "rb") as f:
    saved_vars = pickle.load(f)
    list_of_var = list(saved_vars.keys())
    print(list_of_var)
    
    # Load variables into global namespace
    for var in list_of_var:
        globals()[var] = saved_vars[var]
        print(f"Loaded: {var}")

## Below this is an old code, dont launch

In [ ]:
conn = sqlite3.connect('Enrol_intervals2023.db')
c = conn.cursor()

c.execute("SELECT COUNT(DISTINCT enrolid) FROM ENROL_INTERVAL")
result = c.fetchall()
print(result)

In [ ]:
c.execute("SELECT MIN(startyear), MAX(startyear) FROM ENROL_INTERVAL")
result = c.fetchone()

earliest_year = result[0]
most_recent_year = result[1]

print(f"Earliest start year: {earliest_year}")
print(f"Most recent start year: {most_recent_year}")


In [ ]:
c.execute("""
SELECT COUNT(DISTINCT enrolid) 
FROM ENROL_INTERVAL 
WHERE startyear >= 2003 AND startyear <= 2020
""")
result = c.fetchone()  
patient_count = result[0]

print(f"Patients with enrollment starting 2003-2020: {patient_count:}")

In [ ]:
c.execute("""
SELECT DISTINCT enrolid 
FROM ENROL_INTERVAL 
WHERE startyear >= 2003 
AND startyear <= 2020 
AND days_in_interval >= 365
""")

results2 = c.fetchall()  # Use fetchall() since we want multiple patient IDs


In [ ]:
print(len(results2))

In [ ]:
print(results2[0])

In [ ]:
# Extract patient IDs from tuples into a list
patient_ids = [row[0] for row in results2]
print(f"First 10 patient IDs: {patient_ids[:10]}")

In [ ]:
conn2 = sqlite3.connect('DEMO_2023.db')
c2 = conn2.cursor()

In [ ]:
c2.execute(f"""
SELECT * 
FROM DEMO
""")


In [ ]:
col_names = [desc[0] for desc in c2.description]
result3 = c2.fetchone()

print(col_names)
print(result3)

In [ ]:
c.execute("DROP TABLE IF EXISTS temp_enrollment_with_age")
c.execute("DETACH DATABASE db2")  


In [ ]:
min_age = 20
db2_path = 'DEMO_2023.db'
c.execute(f"""
    CREATE TEMP TABLE temp_enrollment_with_age AS
    SELECT *, 
           NULL as birth_year,
           NULL as age_at_enrollment
    FROM ENROL_INTERVAL
    WHERE startyear >= 2003 
    AND startyear <= 2020 
    AND days_in_interval >= 365
""")

# Attach the second database to access DEMO table
c.execute(f"ATTACH DATABASE '{db2_path}' AS db2")

# Now update using the attached database
c.execute("""
    UPDATE temp_enrollment_with_age
    SET birth_year = (
        SELECT dobyr 
        FROM db2.DEMO 
        WHERE db2.DEMO.enrolid = temp_enrollment_with_age.enrolid
    ),
    age_at_enrollment = startyear - (
        SELECT dobyr 
        FROM db2.DEMO 
        WHERE db2.DEMO.enrolid = temp_enrollment_with_age.enrolid
    )
""")
# Get unique enrolids - patients 20+ at enrollment
c.execute(f"""
    SELECT DISTINCT enrolid
    FROM temp_enrollment_with_age
    WHERE age_at_enrollment >= {min_age}
""")

results = c.fetchall()
print(f"Found {len(results):} unique patients who were {min_age}+ at enrollment")

# Extract just the IDs into a list if needed
eligible_patient_ids = [row[0] for row in results]

conn.commit()
# Clean up
c.execute("DETACH DATABASE db2")
c.execute("DROP TABLE temp_enrollment_with_age")